# 🏠 AeroSync: Dedicated Binary Building Footprint Extraction
**Problem Statement ID: 26012 | DoLR, Ministry of Rural Development**
**Target Task: Ultra-High Precision Building Extraction & Area Estimation**

## Step 0: Auto-Install Required Dependencies

In [ ]:
import sys, subprocess
print('[OK] Dependencies ready.')

## Step 1: Setup & Imports

In [ ]:
import os, sys, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from PIL import Image
import cv2, tifffile

# ── AeroSync workspace resolver (Colab / Kaggle / Local Auto-Detection) ───────
_candidates = [
    os.getcwd(),
    r"C:\AeroSync",
    "/content/AeroSync",
    "/content",
    "/kaggle/working/AeroSync",
    "/kaggle/working",
    os.path.abspath(".."),
]

workspace_dir = next(
    (p for p in _candidates if p and os.path.exists(os.path.join(p, "models"))),
    None,
)

# Auto-clone repository if running in Google Colab / Kaggle / isolated env
if workspace_dir is None:
    print("[INFO] 'models' module not found locally. Auto-cloning AeroSync repository...")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/thatvivekhingu/AeroSync.git"],
            check=True,
        )
        for _p in ["AeroSync", "/content/AeroSync", "/kaggle/working/AeroSync"]:
            if os.path.exists(os.path.join(_p, "models")):
                workspace_dir = os.path.abspath(_p)
                break
    except Exception as _e:
        print(f"[WARNING] Could not auto-clone repository: {_e}")

if workspace_dir is None:
    workspace_dir = os.getcwd()

if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# ── Import upgraded AeroSync v2.0 modules ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from models import (
    AeroSyncAttentionResUNet, AeroSyncUNet,
    FocalDiceCadastralLoss, CombinedCadastralLoss,
    AeroSyncTotalLoss, BoundaryLoss, clDiceLoss,
    mask_to_cadastral_geojson, orthogonalize_polygon, regularize_polygon,
    MCDropoutInference, TTAInference, ProductionInference,
    set_seed, TrainingConfig, ModelEMA,
    CadastralDroneDataset, make_dataloaders,
    decode_mask_to_color,
    CLASS_NAMES, CLASS_COLORS,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] AeroSync v2.0 loaded | workspace: {workspace_dir} | device: {device}")


## Step 2: Binary Class Setup

In [ ]:
BINARY_CLASS_NAMES = {0: 'Background', 1: 'Building Footprint'}
BINARY_CLASS_COLORS = {0: (30, 30, 30), 1: (255, 165, 0)}
print('[OK] Binary Building classes defined.')

## Step 3: Binary Dataset Loader

In [ ]:
search_paths = [r"C:\AeroSync\dataset\Svamitva", "/content/AeroSync/dataset/Svamitva", "/content/dataset/Svamitva", "./dataset/Svamitva"]
dataset_root = next((os.path.abspath(p) for p in search_paths if os.path.exists(p)), os.path.abspath("./dataset/Svamitva"))
images_dir = os.path.join(dataset_root, "FilteredData", "Images")
binary_masks_dir = os.path.join(dataset_root, "FilteredData", "BinaryMasks")
if not os.path.exists(binary_masks_dir) or len(os.listdir(binary_masks_dir)) == 0:
    binary_masks_dir = os.path.join(dataset_root, "FilteredData", "Masks")
img_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.tif'))]) if os.path.exists(images_dir) else []
msk_files = sorted([os.path.join(binary_masks_dir, f) for f in os.listdir(binary_masks_dir) if f.endswith(('.png', '.jpg', '.tif'))]) if os.path.exists(binary_masks_dir) else []
print(f"Binary Dataset: {len(img_files)} Images | {len(msk_files)} Building Masks")

## Step 4: PyTorch Binary Building Dataset

In [ ]:
class BinaryBuildingDataset(Dataset):
    def __init__(self, img_paths, mask_paths, img_size=(512, 512), is_train=True):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.is_train = is_train
    def __len__(self):
        return len(self.img_paths)
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB').resize(self.img_size, Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0
        if idx < len(self.mask_paths) and os.path.exists(self.mask_paths[idx]):
            msk = Image.open(self.mask_paths[idx]).convert('L').resize(self.img_size, Image.NEAREST)
            mask_np = (np.array(msk) > 50).astype(np.int64)
        else:
            mask_np = np.zeros(self.img_size, dtype=np.int64)
        return torch.from_numpy(img_np.copy()).permute(2, 0, 1).float(), torch.from_numpy(mask_np.copy()).long()

train_ds = BinaryBuildingDataset(img_files, msk_files, is_train=True)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
print(f"[OK] Binary Loader ready: {len(train_ds)} samples.")

## Step 5: Binary Model & Loss Setup

In [ ]:
from models import AeroSyncAttentionResUNet, FocalDiceCadastralLoss
model = AeroSyncAttentionResUNet(in_channels=3, num_classes=2, base_filters=32).to(device)
criterion = FocalDiceCadastralLoss(num_classes=2, dice_weight=0.5, focal_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"[OK] Dedicated Binary Building Attention ResUNet initialized on {device}.")

## Step 6: Binary Model Training Loop

In [ ]:
print("Training Binary Building Extraction Model...")
model.train()
epochs = 5
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{epochs} | Loss: {total_loss / max(1, len(train_loader)):.4f}")
print("[OK] Dedicated Binary Building Extractor Trained Successfully.")

## Step 7: Building Extraction & Orthogonalized GeoJSON Export

In [ ]:
from models import mask_to_cadastral_geojson
sample_img_path = img_files[0] if img_files else None
if sample_img_path:
    raw_img = Image.open(sample_img_path).convert('RGB').resize((512, 512))
    inp_t = (torch.from_numpy(np.array(raw_img, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)).to(device)
    model.eval()
    with torch.no_grad():
        pred_bldg = torch.argmax(model(inp_t), dim=1).squeeze(0).cpu().numpy()
    geojson = mask_to_cadastral_geojson(pred_bldg, class_id=1, min_area=15.0)
    out_path = os.path.join(workspace_dir, 'Dedicated_Building_Footprints.geojson')
    with open(out_path, 'w') as f:
        json.dump(geojson, f, indent=2)
    print(f"Extracted {len(geojson['features'])} Building Footprints to {out_path}")